# PopOut Project

## Summary

This project was developed for the Artificial Inteligence course during the first semester of 2026 in the University of Porto. Our main goal was applying coding and critical thinking skills to adapt adversary search algorithms (Monte Carlo Tree Search) and learning algorithms (ID3 Decision Trees). Our project is divided mainly in three parts:

- Creating the game class that includes the game logic (board, moves, rules) and the main loop.

- Implementing the Monte Carlo Tree Search algorithm with the UCT formula for better exploration/exploitation balance.

- Defining the ID3 Decision Tree, its parameters and splits, and training it on existing Iris data and later on generated PopOut game data.

### PopOut - Game Class and loop

PopOut is a variant of the game Connect4. The main goal is to land 4 consecutive pieces of their own colour on a horizontal, vertical or diagonal line. The difference between this variant and the original game consists in a new move, that allows the player to "pop" the piece from the bottom row of the board, if it's one's own colour. This creates new possibilities and increases the game complexity. It also creates a small issue with determining the winner of the game, as by popping out, rows of four pieces can be made for both players simultaneously. This and the following are to be treated on the game loop:

- If a pop move creates wins for both players, the one who made the move wins the game.

- In the case of a full board the current player is allowed to end the game as a draw instead of making a pop move.

- If the board state is repeated 3 times, the game ends as a draw.

#### Board Structure

Instead of a regular matrix, we looked for a better, more optimized format and we reached the conclusion that using a bitboard was the most efficient choice.
Therefore the board is defined as a 64 bit, 49 long integer (49 instead of 42 because we have a buffer row that separates the columns on the bitboard), that allows faster and lower cost operations, using shifts, additions, and logic operations such as "OR", "AND", "XOR", "NOT" to combine the board states and update the game. Alongside this measure, we allocated every board state and its mask (that is used in a "XOR" operation to differentiate each player's pieces) an identifier to create a HashMap. This data structure allows for an easier draw identification and facilitates our MCTS algorithm to access previously generated states.

In [ ]:
from random_play import RandomPlay
from mcts_play import MCTSPlay
from id3_play import ID3Play

class Connect4:
    """
    Main class to manage the game initialization
    """
    def __init__(self):
        """
        Initializes the Connect4 game.
        """
        self.turn = 0
        self.result = None
        self.terminal = False
        
    def get_initial_position(self):
        """
        Gets the starting position of the game.

        Returns:
            Position: The initial board state with turn 0.
        """
        return Position(self.turn)
                
class Position:
    """
    Represents a state of the game board using bitboards.
    """
    # Removed 'history' parameter to avoid memory overload during rollouts
    def __init__(self, turn, mask = 0, position = 0, num_turns = 0):
        """
        Initializes a game state.

        Args:
            turn (int): The current player's turn (0 or 1).
            mask (int): Bitboard representing all pieces on the board. Defaults to 0.
            position (int): Bitboard representing the current player's pieces. Defaults to 0.
            num_turns (int): Counter for the number of turns played. Defaults to 0.
        """
        self.turn = turn
        self.result = None
        self.terminal = False
        self.num_turns = num_turns
        self.mask = mask
        self.position = position
        self._compute_hash()
                
    def move(self, loc):
        """
        Applies a move and returns a new board state.

        Args:
            loc (int): The chosen move. 0-6 for dropping a piece, 7-13 for popping a piece from the bottom, or -1 for a draw.

        Returns:
            Position: A new Position instance of the board after the move.
        """
        # Rules 2 and 3, draw
        if loc == -1:
            # Creation without history copy (pure math and bitwise operations)
            new_pos = Position(int(not self.turn), self.mask, self.position ^ self.mask, self.num_turns + 1)
            new_pos.terminal = True
            new_pos.result = 0
            return new_pos

        is_pop = False
        
        if 0 <= loc <= 6:
            # Normal move (0-6)
            new_position = self.position ^ self.mask
            new_mask = self.mask | (self.mask + (1 << (loc*7)))
        else:
            # Pop move (7-13)
            c = loc - 7
            col_mask = 0b111111 << (c * 7)
            
            curr_col = self.position & col_mask
            opp_col = (self.position ^ self.mask) & col_mask
            
            # shift move
            new_curr_col = (curr_col >> 1) & col_mask
            new_opp_col = (opp_col >> 1) & col_mask
            
            new_curr = (self.position & ~col_mask) | new_curr_col
            new_opp = ((self.position ^ self.mask) & ~col_mask) | new_opp_col
            
            new_position = new_opp
            new_mask = new_curr | new_opp
            is_pop = True

        # mathematical instance creation (tries to fix garbage collection bottleneck)
        new_pos = Position(int(not self.turn), new_mask, new_position, self.num_turns + 1)
        new_pos.game_over(is_pop)
        return new_pos
    
    # return list of legal moves
    def legal_moves(self):
        """
        Determines all valid moves for the current board state.

        Returns:
            list: A list of integers representing valid moves (0-6 for drops, 7-13 for pops, -1 for draw).
        """
        bit_moves = []
        # regular moves
        for i in range(7):
            col_mask = 0b111111 << 7 * i
            if col_mask != self.mask & col_mask:
                bit_moves.append(i)
                
        # pop move
        # checks if the piece belongs to the current player
        for i in range(7):
            bottom_bit = 1 << (i * 7)
            if self.position & bottom_bit:
                bit_moves.append(i + 7)

        # draw check, full board
        is_full = (self.mask == 279258638311359)
        
        if is_full:
            bit_moves.append(-1)
            
        return bit_moves

    def game_over(self, is_pop=False):
        """
        Checks if the game has ended and updates the terminal status and result.

        Args:
            is_pop (bool, optional): Indicates if the last move was a PopOut move. Defaults to False.
        """
        # sets result to -1, 0, or 1 if game is over (otherwise self.result is None)
        just_moved_won = self.connected_four_fast()
        about_to_move_won = False
        
        if is_pop:
            about_to_move_won = self.connected_four_fast(self.position)
            
        if just_moved_won or about_to_move_won:
            self.terminal = True
            
            if just_moved_won:
                self.result = 1 if self.turn == 1 else -1 
            else:
                self.result = 1 if self.turn == 0 else -1
        else:
            self.terminal = False
            self.result = None
            
        if not self.terminal and not self.legal_moves():
            self.terminal = True
            self.result = 0
            
    def connected_four_fast(self, board_position=None):
        """
        Detection of four connected pieces using bitwise shifts.

        Args:
            board_position (int): The board to evaluate. If None, evaluates the current player's board.

        Returns:
            bool: True if there is a 4-in-a-row, False otherwise.
        """
        if board_position is None:
            board_position = self.position ^ self.mask
            
        # Horizontal check
        m = board_position & (board_position >> 7)
        if m & (m >> 14):
            return True
        # Diagonal \
        m = board_position & (board_position >> 6)
        if m & (m >> 12):
            return True
        # Diagonal /
        m = board_position & (board_position >> 8)
        if m & (m >> 16):
            return True
        # Vertical
        m = board_position & (board_position >> 1)
        if m & (m >> 2):
            return True
        # Nothing found
        return False
    
    
    def _compute_hash(self):
        """
        Computes a unique hash for the current board state.
        """
        position_1 = self.position if self.turn == 0 else self.position ^ self.mask
        self.hash = 2 * hash((position_1, self.mask)) + self.turn
    
    def __hash__(self):
        return self.hash
        
    def __eq__(self, other):
        return isinstance(other, Position) and self.turn == other.turn and self.mask == other.mask and self.position == other.position

    # Visual interface
    def print_board(self, legal_moves=None):
        """
        Displays the board to the console.

        Args:
            legal_moves (list): List of legal moves to be displayed for the player.
        """
        p0_bits = self.position if self.turn == 0 else (self.position ^ self.mask)
        p1_bits = self.position if self.turn == 1 else (self.position ^ self.mask)
        
        display = "\n  7  8  9 10 11 12 13 : Pop"
        display += "\n  0  1  2  3  4  5  6 : Drop\n-----------------------"
        for row in range(5, -1, -1):
            line = "|"
            for col in range(7):
                bit = 1 << (col * 7 + row)
                if p0_bits & bit:
                    line += " O " # Player 0
                elif p1_bits & bit:
                    line += " X " # Player 1
                else:
                    line += " . " # Empty
            display += "\n" + line + "|"
        display += "\n-----------------------"

        if legal_moves is not None:
                    drops = [m for m in legal_moves if 0 <= m <= 6]
                    pops = [m for m in legal_moves if 7 <= m <= 13]
                    draw = [-1] if -1 in legal_moves else []
                    
                    display += "\nLegal Moves:"
                    if drops: display += f"\n  Drops: {drops}"
                    if pops:  display += f"\n  Pops:  {pops}"
                    if draw:  display += f"\n  Draw:  [-1]"
                    display += "\n"

        print(display)

# Game loop
if __name__ == "__main__":
    
    def choose_player(num_player):
        """
        Displays a menu for the user to select the type of agent for a specific player.

        Args:
            num_player (int): The player ID (0 or 1).

        Returns:
            object: An instance of the selected player class (HumanPlay, RandomPlay, MCTSPlay, or ID3Play).
        """
        while True:
            print(f"\nChoose a player for {num_player}")
            print("1 - Human")
            print("2 - Random Play")
            print("3 - MCTS")
            print("4 - ID3 (MCTS)")
            player_str = input("Enter 1 or 2 or 3 or 4: ").strip()
            
            if player_str == '1':
                return HumanPlay(f"Player {num_player} (Human)")
            elif player_str == '2':
                return RandomPlay(f"Player {num_player} (Random)")
            elif player_str == '3':
                return MCTSPlay(f"Player {num_player} (MCTS)", time_limit=2.0)
            elif player_str == '4':
                return ID3Play(f"Player {num_player} (ID3)")
            else:
                print("Invalid player! Please enter 1, 2 or 3.")

    agents = {
        0: choose_player(0),
        1: choose_player(1)
    }

    game = Connect4()
    pos = game.get_initial_position()
    
    print(f"\n--- Game start: {agents[0].name} VS {agents[1].name} ---")

    # memory stored to check for draws
    global_history = {}

    # Main Loop
    while not pos.terminal:
        # updates the current board state occurences
        global_history[pos.hash] = global_history.get(pos.hash, 0) + 1
        
        legal = pos.legal_moves()
        
        # Injects the draw option (-1) if threefold repetition is met
        if global_history[pos.hash] >= 3 and -1 not in legal:
            legal.append(-1)
            print("\n[!] Attention: This state has repeated 3 times. Draw (-1) is available.")

        pos.print_board(legal_moves=legal)
        
        current_agent = agents[pos.turn]
        print(f"\n{current_agent.name}'s turn.")
        
        move = current_agent.get_move(pos)
        pos = pos.move(move)

    # End of game
    pos.print_board()
    if pos.result == 0:
        print("It's a Draw!")
    else:
        winner = 0 if pos.result == 1 else 1
        print(f"Player {winner} wins!")


Choose a player for 0
1 - Human
2 - Random Play
3 - MCTS
4 - ID3 (MCTS)

Choose a player for 1
1 - Human
2 - Random Play
3 - MCTS
4 - ID3 (MCTS)

--- Game start: Player 0 (Random) VS Player 1 (MCTS) ---

  7  8  9 10 11 12 13 : Pop
  0  1  2  3  4  5  6 : Drop
-----------------------
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
-----------------------
Legal Moves:
  Drops: [0, 1, 2, 3, 4, 5, 6]


Player 0 (Random)'s turn.
Player 0 (Random) played: 4

  7  8  9 10 11 12 13 : Pop
  0  1  2  3  4  5  6 : Drop
-----------------------
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  O  .  . |
-----------------------
Legal Moves:
  Drops: [0, 1, 2, 3, 4, 5, 6]


Player 1 (MCTS)'s turn.
[Player 1 (MCTS)] Thinking for 2.0 seconds using 4 cores...


### Human input collection

A simple prompt to read human player's moves.

In [ ]:
class HumanPlay:
    """
    Wrapper class that manages the human player input.

    Attributes:
        name (str): The display name of the human player.
    """
    def __init__(self, name="Human"):
        """
        Initializes the instance
        """
        self.name = name

    def get_move(self, position):
        """
        Prompts the user for a move and validates it against the legal moves.

        Args:
            position : The current state of the game board.

        Returns:
            int: The index of the selected valid move.
        """
        legal = position.legal_moves()
        while True:
            try:
                move = int(input(f"[{self.name}] Choose a move : "))
                if move in legal:
                    return move
                print("Invalid move! Try again.")
            except ValueError:
                print("Please enter a number.")

### Random play

Play's a random move from the legal action list. It's slowed down for better readability for the user.

In [ ]:
import random
import time

class RandomPlay:
    """
    Wrapper class that manages an agent that plays completely random moves.

    Attributes:
        name (str): The display name of the random agent.
    """
    def __init__(self, name="Random"):
        """
        Initializes the RandomPlay instance.

        Args:
            name (str, optional): The name of the agent. Defaults to "Random".
        """
        self.name = name

    def get_move(self, position):
        """
        Evaluates the board state and returns a random legal move.

        Args:
            position (Position): The current state of the game board.

        Returns:
            int: The index of the randomly selected valid move.
        """
        legal_moves = position.legal_moves()
        
        time.sleep(0.5) # Doesn't let the game play too fast 
        
        move = random.choice(legal_moves)
        print(f"{self.name} played: {move}")
        return move

### Monte Carlo Tree Search

The Monte Carlo Tree Search is a search algorithm used to build and traverse very big trees, by selecting nodes, expanding their leafs, simulating random games (rollouts), and backpropagating its results to evaluate the nodes before starting the cycle once again.

Our biggest deviation from the regular implementation of MCTS is using a dictonary based graph instead of a Node class. That is possible thanks to our hashed board state wich allows using it as a key for each "node" and also permits the tree to avoid recreating the same structures, by recognizing transpositions. This reduces memory use and lightens the program, as no classes are created at every iteration. A few of our design choices:

- Max_depth = 50: Our rollouts are capped at a 50 total move (both players added) depth. This number was chosen because to cover the total board, only 42 pieces are needed (42 moves) and the "pop moves" potentialy allow the game to run infinitely. When that max is reached, a draw is rewarded.

- Num_runs = 10: Unlike regular MCTS that performs a single random rollout game per node expansion, we decided to execute 10 rollouts everytime. This permits the algorithm to catch a wider view of the board state situation and properly rewards the play instead of relying on lucky wins.

- UCB exploration_const = 2.0: The constant is a default choice for most applications and served us well in exploring different branches of our tree.

- Parallelism: To reach higher speeds (at peak 50k itterations and 500k+ rollouts per second) we implemented a multiprocessing pool that distributes workload to different CPU cores (4 was the vallue used) allowing multiple search trees to be built and evaluated simultaneously, merging the statistics from all root children at the end of the time limit.

In [4]:
import random
import math
import time
import os
from multiprocessing import Pool

def mcts_worker(args):
    """
    Worker function to build an MCTS tree in a separate CPU core.

    Args: tuple containing root_state and time_limit.

    Return: tuple containing children_stats (dict), iterations (int) and total rollouts (int).
    """
    root_state, time_limit = args
    
    # Ensures each CPU core generates completely different random games
    random.seed(os.urandom(4))

    nodes, iterations, total_rollouts = build_mcts_tree(root_state, time_limit)
    
    children_stats = {}
    for loc in root_state.legal_moves():
        next_state = root_state.move(loc)
        if next_state in nodes:
            w, n, _ = nodes[next_state]
            children_stats[loc] = (w, n)
            
    return children_stats, iterations, total_rollouts


def build_mcts_tree(root_state, time_limit):
    """
    Builds the MCTS statistical tree given a time limit.

    Args: tuple containgin root_state and time_limit.

    Return: tuple containing nodes (dict), iterations (int) and total_rollouts (int).
    """
    # Structure: {state: (total_reward, visits, {parent_state: parent_visits})}
    nodes = {} 
    nodes[root_state] = (0.0, 0.0, {root_state: 0})
    
    start_time = time.time()
    iterations = 0
    total_rollouts = 0 
    
    while time.time() - start_time < time_limit:   
        iterations += 1

        # ==========================================
        # SELECTION
        # ==========================================
        selection_path = selection_phase(nodes, root_state)
        selected_node = selection_path[-1]
        
        _, visits, _ = nodes[selected_node]
        
        # ==========================================
        # EXPANSION
        # ==========================================
        if visits > 0 and not selected_node.terminal:
            legal_moves = selected_node.legal_moves()
            for loc in legal_moves:
                new_state = selected_node.move(loc)
                if new_state not in nodes:
                    nodes[new_state] = (0.0, 0.0, {selected_node: 0})
                    
            loc = random.choice(legal_moves)
            child_state = selected_node.move(loc)
            
            # ==========================================
            # SIMULATION / ROLLOUT
            # ==========================================
            reward = 0
            num_runs = 10
            for _ in range(num_runs):
                reward += simulate_rollout(child_state)
            total_rollouts += num_runs
            
            w, n, parent_n_dict = nodes[child_state]
            if selected_node not in parent_n_dict:
                parent_n_dict[selected_node] = 0
            parent_n_dict[selected_node] += 1
            nodes[child_state] = (w + reward, n + num_runs, parent_n_dict)

        else:
            # ==========================================
            # SIMULATION / ROLLOUT (Direct Simulation)
            # ==========================================
            reward = 0
            num_runs = 10
            for _ in range(num_runs):
                reward += simulate_rollout(selected_node)   
            total_rollouts += num_runs
        
        # ==========================================
        # BACKPROPAGATION
        # ==========================================
        parent = root_state
        for state_in_path in selection_path:
            w, n, parent_n_dict = nodes[state_in_path]
            parent_n_dict[parent] += num_runs
            nodes[state_in_path] = (w + reward, n + num_runs, parent_n_dict)
            parent = state_in_path
            
    return nodes, iterations, total_rollouts


def mcts_agent(time_limit, num_workers=4):
    """
    Wraps the MCTS strategy, returning a move function.

    Args: tuple containing time_limit and num_workers.

    Return: function that takes current_state and returns the optimal move.
    """
    def strat(current_state):
        # Prepare arguments for multiprocessing
        args = [(current_state, time_limit) for _ in range(num_workers)]
        
        # Run MCTS in parallel across available CPU cores
        with Pool(processes=num_workers) as pool:
            results = pool.map(mcts_worker, args)
            
        total_iters = 0
        total_rollouts = 0
        merged_stats = {}
        
        # Merge stats from all workers
        for children_stats, iters, rollouts in results:
            total_iters += iters
            total_rollouts += rollouts
            
            for loc, (w, n) in children_stats.items():
                if loc not in merged_stats:
                    merged_stats[loc] = [0.0, 0.0]
                merged_stats[loc][0] += w
                merged_stats[loc][1] += n

        iter_per_sec = total_iters / time_limit
        rollouts_per_sec = total_rollouts / time_limit
        
        print(f"Number of MCTS iterations: {total_iters} ({iter_per_sec:.0f} iters/sec) [Cores: {num_workers}]")
        print(f"Number of MCTS rollouts:   {total_rollouts} ({rollouts_per_sec:.0f} rollouts/sec) [Cores: {num_workers}]")
        
        player = current_state.turn
        best_score = float('-inf') if player == 0 else float('inf')
        next_best_move = None
   
        for loc in current_state.legal_moves():
            if loc not in merged_stats:
                score = 0.0
            else:
                w, n = merged_stats[loc]
                score = 0.0 if n == 0 else w / n
            
            if score < best_score and player == 1: # Player 1 minimizes
                best_score = score
                next_best_move = loc
            elif score > best_score and player == 0: # Player 0 maximizes
                best_score = score
                next_best_move = loc
                
        # Safety fallback
        if next_best_move is None:
            return random.choice(current_state.legal_moves())
            
        return next_best_move
    return strat
        

def simulate_rollout(state, max_depth=50):
    """
    Executes random rollout from given state until the game ends or reaches the number of plays determined by max_depoth.

    Args: tuple containing state and max_depth.

    Return: float for standardized reward: 1.0 for player 0 win, 0.0 for player 1 win, and 0.5 for a draw.
    """
    current_state = state
    depth = 0
    
    while not current_state.terminal and depth < max_depth:
        moves = current_state.legal_moves()
        loc = random.choice(moves)
        current_state = current_state.move(loc) 
        depth += 1
        
    if not current_state.terminal:
        return 0.5 

    if current_state.result == 1: return 1.0
    elif current_state.result == -1: return 0.0
    return 0.5
        

def selection_phase(nodes, root_state):
    """
    Navigates the tree using the UCB1 policy.

    Args: tuple containing nodes and root_state.

    Return: a list containing the path of traversed nodes from the root to the selected leaf.
    """
    current_node = root_state
    path = []
    visited = set() # Cycle detector for PopOut infinite loops
    
    while True:
        if current_node in visited:
            return path
        
        visited.add(current_node)
        
        _, visits, _ = nodes[current_node]
        path.append(current_node)
        
        if visits == 0:
            return path
        
        legal_moves = current_node.legal_moves()
        next_player = current_node.turn
        
        best_score = float('-inf') if next_player == 0 else float('inf')
        next_best_node = None
        
        for loc in legal_moves:
            result_state = current_node.move(loc)
            
            if result_state not in nodes:
                return path
                
            temp_w, temp_visits, temp_parent_n_count = nodes[result_state]
            if current_node not in temp_parent_n_count:
                temp_parent_n_count[current_node] = 0
                
            if temp_parent_n_count[current_node] == 0:
                path.append(result_state)
                return path
                        
            score = calculate_ucb_score(nodes[current_node][1], temp_parent_n_count[current_node], temp_w / temp_visits, next_player)
            
            if score < best_score and next_player == 1:
                best_score = score
                next_best_node = result_state
            elif score > best_score and next_player == 0:
                best_score = score
                next_best_node = result_state
                
        current_node = next_best_node
        if current_node is None:
            return path
            
            
def calculate_ucb_score(parent_visits, node_visits, win_rate, player, exploration_const=2.0):
    """
    Upper Confidence Bound (UCB1) formula.

    Args: tuple containg parent_visits, node_visits, win_rate, player and exploration_const

    Return: a float of the calculated UCB1 value
    """
    exploration_term = math.sqrt(exploration_const * math.log(parent_visits) / node_visits)
    if player == 0: 
        return win_rate + exploration_term
    return win_rate - exploration_term


class MCTSPlay:
    """
    Wrapper class that manages the MCTS player.

    Atributtes:
        name (str): the display name of the agent.
        time_limit (float): maximum calculation time per move.
        num_workers (int): number of parallel processes to use.
        strategy (function): the strategic move evaluation function.
    """
    def __init__(self, name="MCTS", time_limit=2.0, num_workers=4):
        """Initializes the MCTSPlay instance with custom settings.

        Args: tuple containing name, time_limit and num_workers.
        """
        self.name = name
        self.time_limit = time_limit
        self.num_workers = num_workers
        self.strategy = mcts_agent(self.time_limit, self.num_workers)

    def get_move(self, position):
        """
        Evaluates the board state and returns the optimal move.
        
        Args: position.

        Return: the index of the selected play (int)
        """
        print(f"[{self.name}] Thinking for {self.time_limit} seconds using {self.num_workers} cores...")
        move = self.strategy(position)
        print(f"{self.name} chose to play: {move}")
        return move

### ID3 Decision Tree

The ID3 is a supervised learning algorithm that constructs a decision tree based on historical training data. The algorithm utilizes the concept of Shannon Entropy to measure the impurity or uncertainty within a dataset. For each level of the tree, the ID3 engine evaluates all available features and selects the one that best splits the target classes, maximizing information gain. Its goal is to translate complex data into a fast, readable hierarchy of logical rules that can be evaluated much more quickly.

We implemented the following architectural choices:

- Dynamic Numerical Discretization: Standard ID3 algorithms are designed for categorical data. To handle continuous numerical feature arrays, we implemented a splitting mechanism. The engine dynamically extracts unique values from each feature column, calculates intermediate midpoints (thresholds), and generates optimal binary splits (<= threshold and > threshold). This dynamic approach allows the engine to process numerical datasets while keeping the tree structure compact and efficient.

- Depth_limit: Decision trees are prone to overfitting, meaning they tend to memorize the training data perfectly but fail to generalize unseen examples. To combat this, we integrated a depth_limit parameter into the engine. This acts as a regularization tool, forcing the tree to learn the fundamental patterns and logical boundaries of the dataset instead of memorizing exact data configurations.

- Leaf Node Class Labeling: Once the recursive tree-building process (_build_tree) reaches a completely pure state, or hits the defined depth limit, it generates a terminal DecisionNode (a leaf). If the data remaining at a leaf is still mixed due to the depth constraint, the algorithm employs a majority-vote fallback mechanism, assigning the most statistically common target class to that leaf to ensure reliable predictions.

In [ ]:
import numpy as np
from collections import Counter
import math
import pandas as pd
from id3_engine import ID3DecisionTree

class DecisionNode:
    """
    Represents a single node or leaf in the decision tree.
    """
    def __init__(self, left=None, right=None, feature_idx=None, threshold=None, class_label=None):
        """
        Initializes a DecisionNode.

        Args:
            left (DecisionNode): The left branch (child node). Defaults to None.
            right (DecisionNode): The right branch (child node). Defaults to None.
            feature_idx (int): The index of the feature (board cell) used for splitting. Defaults to None.
            threshold (float): The value used to split the data. Defaults to None.
            class_label (int): The final move prediction (only for leaves). Defaults to None.
        """
        self.left = left            
        self.right = right          
        self.feature_idx = feature_idx  
        self.threshold = threshold      
        self.class_label = class_label  

    def is_leaf(self):
        """
        Checks if the current node is a leaf node (endpoint).

        Returns:
            bool: True if the node has a final class prediction, False otherwise.
        """
        return self.class_label is not None

class ID3DecisionTree:
    """
    Implementation of the ID3 Decision Tree algorithm with numerical discretization support.
    
    Attributes:
        depth_limit (int/float): The maximum allowed depth for the tree to prevent overfitting.
        root (DecisionNode): The root node of the trained tree.
    """
    def __init__(self, depth_limit=float('inf')):
        """
        Initializes the ID3DecisionTree instance.

        Args:
            depth_limit (int or float): Maximum depth limit. Defaults to infinity.
        """
        self.root = None
        self.depth_limit = depth_limit 

    def _entropy(self, y):
        """
        Calculates the entropy of a given target array. 
        Higher entropy means the target classes are very mixed.

        Args:
            y (numpy.ndarray): Array of target labels.

        Returns:
            float: The calculated entropy value.
        """
        if len(y) == 0: return 0
        counts = Counter(y)
        probs = [counts[c] / len(y) for c in counts]
        return -sum(p * math.log2(p) for p in probs if p > 0)

    def _information_gain(self, y, y_left, y_right):
        """
        Calculates how much information is gained by a specific split.

        Args:
            y (numpy.ndarray): Original array of labels before the split.
            y_left (numpy.ndarray): Array of labels for the left split.
            y_right (numpy.ndarray): Array of labels for the right split.

        Returns:
            float: The calculated information gain.
        """
        parent_ent = self._entropy(y)
        n = len(y)
        n_l, n_r = len(y_left), len(y_right)
        if n_l == 0 or n_r == 0: return 0
        
        child_ent = (n_l / n) * self._entropy(y_left) + (n_r / n) * self._entropy(y_right)
        return parent_ent - child_ent

    def _best_split(self, X, y):
        """
        Iterates through all features to find the optimal split point (highest information gain).

        Args:
            X (numpy.ndarray): The feature matrix.
            y (numpy.ndarray): The target labels array.

        Returns:
            tuple: A tuple containing (best_feature_index, best_threshold). Returns (None, None) if no valid split.
        """
        best_gain = -1
        split_idx, split_thresh = None, None

        for idx in range(X.shape[1]):
            column_values = X[:, idx]
            unique_values = np.unique(column_values)
            thresholds = (unique_values[:-1] + unique_values[1:]) / 2

            for t in thresholds:
                left_mask = column_values <= t
                y_l, y_r = y[left_mask], y[~left_mask]
                
                gain = self._information_gain(y, y_l, y_r)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = idx
                    split_thresh = t
        
        return split_idx, split_thresh

    def fit(self, X, y):
        """
        Starts the tree building process using the provided training data.

        Args:
            X (numpy.ndarray): The training feature matrix.
            y (numpy.ndarray): The training target labels array.
        """
        self.root = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        """
        Recursively builds the decision tree level by level.

        Args:
            X (numpy.ndarray): The feature matrix for the current node.
            y (numpy.ndarray): The target labels array for the current node.
            depth (int): The current depth in the tree. Defaults to 0.

        Returns:
            DecisionNode: The root node of the constructed (sub)tree.
        """
        if len(set(y)) == 1:
            return DecisionNode(class_label=list(y)[0])
        
        if depth >= self.depth_limit or len(y) <= 1:
            most_common = Counter(y).most_common(1)[0][0]
            return DecisionNode(class_label=most_common)

        idx, thresh = self._best_split(X, y)
        
        if idx is None:
            return DecisionNode(class_label=Counter(y).most_common(1)[0][0])

        left_mask = X[:, idx] <= thresh
        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        
        return DecisionNode(left, right, idx, thresh)

    def predict(self, X):
        """
        Predicts the best move for a given set of board states.

        Args:
            X (numpy.ndarray): The feature matrix to predict.

        Returns:
            numpy.ndarray: An array containing the predicted move for each state.
        """
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        """
        Walks down the tree based on feature values until it reaches a leaf.

        Args:
            x (numpy.ndarray): A single state (feature vector).
            node (DecisionNode): The current node in the traversal.

        Returns:
            int: The predicted class label (move).
        """
        if node.is_leaf():
            return node.class_label
        
        if x[node.feature_idx] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

    def show(self, node=None, spacing="", feature_names=None):
        """
        Displays the visual structure of the tree in the terminal.

        Args:
            node (DecisionNode): The starting node. Defaults to the root.
            spacing (str): The string used for indentation.
            feature_names (list): List of feature names for readability. Defaults to None.
        """
        if node is None:
            node = self.root

        if node.is_leaf():
            print(spacing + "  Prediction:", node.class_label)
            return

        feat = feature_names[node.feature_idx] if feature_names else f"Column {node.feature_idx}"
        
        print(f"{spacing}[{feat} <= {node.threshold:.2f}]")

        print(spacing + " ├── Yes:")
        self.show(node.left, spacing + " │   ", feature_names)

        print(spacing + " └── No:")
        self.show(node.right, spacing + "     ", feature_names)

### ID3 Train Test

To validate our Tree Engine, we implemented a training and testing pipeline for processing both the Iris and our PopOut MCTS-generated datasets.

A few of our features are:

- Data Splitting & Reproducibility: To assess how well the tree generalizes to unseen data, we added a data splitting that shuffles and divides datasets into an 80% training set and a 20% testing set. By utilizing a fixed random seed (random_state=42), we assure that the data shuffling is deterministic, allowing for verifiable accuracy metrics.

- PopOut Pipeline: When learning from the MCTS-generated dataset, we must avoid losing plays. Therefore, we had to apply a pre-processing data filter that purges all moves made by the losing player, ensuring the tree is trained exclusively on the exact sequence of moves that lead to a victory.

In [ ]:
def custom_train_test_split(X, y, test_size=0.2, random_state=42):
    """
    Splits the dataset into training and testing sets.

    Args:
        X (numpy.ndarray): The feature matrix of the dataset.
        y (numpy.ndarray): The target labels array.
        test_size (float): The proportion of the dataset used for testing.
        random_state (int): Ensures reproducibility. Sets a fixed number for the seed.

    Returns:
        tuple: A tuple containing four arrays: (X_train, X_test, y_train, y_test).
    """
    np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    X_shuffled, y_shuffled = X[indices], y[indices]
    
    cutoff = int((1 - test_size) * len(X))
    return X_shuffled[:cutoff], X_shuffled[cutoff:], y_shuffled[:cutoff], y_shuffled[cutoff:]

def iris_train():
    """
    Pipeline to train and evaluate the ID3 Decision Tree using the fixed Iris dataset.

    This function loads the 'iris.csv' data, splits it into training and testing sets,
    trains the ID3 model with a depth limit of 4, evaluates the accuracy on both sets,
    visually displays the generated tree, and runs a test case.
    """
    print("\n" + "="*40)
    print("        Training: Iris Dataset")
    print("="*40)
    try:
        df = pd.read_csv('iris.csv')
        X = df.iloc[:, 1:5].values
        y = df.iloc[:, 5].values
        feature_names = df.columns[1:5].tolist()

        # Train/test divison
        X_train, X_test, y_train, y_test = custom_train_test_split(X, y, test_size=0.2)

        print(f"\nTraining tree (Depth: 4) with {len(X_train)} examples...")
        tree = ID3DecisionTree(depth_limit=4)
        
        tree.fit(X_train, y_train)

        acc_train = np.mean(tree.predict(X_train) == y_train) * 100
        acc_test = np.mean(tree.predict(X_test) == y_test) * 100

        print("\n--- Results ---")
        print(f"Accuracy on train set: {acc_train:.2f}%")
        print(f"Accuracy no test set: {acc_test:.2f}%\n")

        print("--- Tree Structure ---")
        tree.show(feature_names=feature_names)

        print("\n--- Single test ---")
        example = np.array([[5.1, 3.5, 1.4, 0.2]])
        print(f"Entry data: {example[0]}")
        print(f"Tree prediction: {tree.predict(example)[0]}")
        
    except FileNotFoundError:
        print("Error: iris.csv not found!")

def popout_train():
    """
    Pipeline to train and evaluate the ID3 Decision Tree using any PopOut dataset.

    Prompts the user for the dataset filename, filters the data to keep
    only the winning moves, splits it into training and testing sets, trains a deeper 
    tree (depth 20) to capture game logic, and evaluates its accuracy without printing 
    the massive tree structure to the terminal.
    """
    print("\n" + "="*40)
    print("       Training: PopOut Dataset")
    print("="*40)
    nome_csv = input("Enter the PopOut csv name (e.g. mcts_2sec_vs_random_10k.csv): ").strip()
    
    try:
        df = pd.read_csv(nome_csv)
        total_original = len(df)
        
        # Filter only the moves from the winner
        filtro_vitorias = (
            ((df['player_turn'] == 0) & (df['game_result'] == 1)) |
            ((df['player_turn'] == 1) & (df['game_result'] == -1))
        )
        df_limpo = df[filtro_vitorias]
        
        print(f"\n[INFO] Total moves on CSV: {total_original}")
        print(f"[INFO] Filtered winner moves: {len(df_limpo)}.")
        
        X = df_limpo.iloc[:, 0:43].values
        y = df_limpo.iloc[:, 43].values
        
        X_train, X_test, y_train, y_test = custom_train_test_split(X, y, test_size=0.2)
        
        print(f"\nTraining tree (Depth: 20) with {len(X_train)} examples...")
        tree = ID3DecisionTree(depth_limit=20)
        tree.fit(X_train, y_train)
        
        acc_train = np.mean(tree.predict(X_train) == y_train) * 100
        acc_test = np.mean(tree.predict(X_test) == y_test) * 100
        
        print("\n--- Results ---")
        print(f"Accuracy on train set: {acc_train:.2f}%")
        print(f"Accuracy on test set: {acc_test:.2f}%")
        print("The tree is too big to be printed on the terminal")
        
    except FileNotFoundError:
        print(f"Error: {nome_csv} not found!")

if __name__ == "__main__":
    while True:
        print("\n" + "="*40)
        print("    Training Menu")
        print("="*40)
        print("1 - Train and visualize Iris")
        print("2 - Train and evaluate PopOut (MCTS)")
        print("0 - Exit")
        
        escolha = input("Escolha uma opção: ").strip()
        
        if escolha == '1':
            iris_train()
        elif escolha == '2':
            popout_train()
        elif escolha == '0':
            print("Exiting...")
            break
        else:
            print("Invalid option!")

### ID3 Play

The ID3Play class functions applies the decision tree engine to game environment. It acts as a bot during live gameplay, deploying the knowledge extracted from the MCTS simulations.

A few of the implemented mechanisms:

- Training and Validation: Upon start, the agent loads the dataset, filters for winning moves, and builds its decision tree (depth_limit=20). It also validates its own reliability by running a 80/20 Holdout evaluation and a 5-Fold Cross-Validation cycle.

- State Translation: Since the game engine operates Bitboards the agent's get_move acts as a decoder, mapping the active bitmasks back into a flat 43-element array (42 board cells encoded as -1 for empty, 0 for Player 0, and 1 for Player 1, plus the current turn index) to feed into the prediction model.

In [ ]:
import pandas as pd
import numpy as np
import random
from id3_engine import ID3DecisionTree
from id3_train_test import custom_train_test_split 

class ID3Play:
    """
    Wrapper class that utilizes the id3 engine to return a move based on the training data.
    
        - loads a dataset of MCTS games
        - filters for winning moves
        - trains a decision tree and uses it to predict optimal moves

    Attributes:
        name (str): The display name of the agent.
        csv_path (str): Path to the dataset used for training.
        tree (ID3DecisionTree): The decision tree model.
    """
    def __init__(self, name="ID3-Bot", csv_path='mcts_2sec_vs_random_10k.csv'):
        """
        Initializes the instance and starts the learning phase.

        Args:
            name (str): The name of the agent, ID3-Bot.
            csv_path (str): The dataset filename. Defaults to 'mcts_2sec_vs_random_10k.csv'.
        """
        self.name = name
        # Initialize the tree with a depth limit to prevent overfitting
        self.tree = ID3DecisionTree(depth_limit=20)
        self.csv_path = csv_path
        self._prepare_model()

    def _prepare_model(self):
        """
        Loads data, filters for optimal plays, and runs validation tests.
        """
        print(f"\n[!] {self.name} learning MCTS gameplay data...")
        try:
            df = pd.read_csv(self.csv_path)
            
            filtro_vitorias = (
                ((df['player_turn'] == 0) & (df['game_result'] == 1)) |
                ((df['player_turn'] == 1) & (df['game_result'] == -1))
            )
            df_limpo = df[filtro_vitorias]
            
            X = df_limpo.iloc[:, 0:43].values  # Board state + player turn
            y = df_limpo.iloc[:, 43].values    # The move chosen by MCTS (target)
            
            X_train, X_test, y_train, y_test = custom_train_test_split(X, y, test_size=0.2)
            
            self.tree.fit(X_train, y_train)
            
            # Check accuracy on unseen data
            preds = self.tree.predict(X_test)
            accuracy = np.mean(preds == y_test) * 100
            print(f"80/20 Holdout Accuracy: {accuracy:.2f}%")

            # 5-FOLD CROSS-VALIDATION
            print("Running 5-Fold Cross-Validation for robustness...")
            k = 5
            fold_size = len(X) // k
            cv_scores = []
            
            for i in range(k):
                start, end = i * fold_size, (i + 1) * fold_size
                
                # Separate one slice for validation and the other 4 for training
                X_val, y_val = X[start:end], y[start:end]
                X_train_cv = np.concatenate([X[:start], X[end:]])
                y_train_cv = np.concatenate([y[:start], y[end:]])
                
                # Create a temporary tree for this specific fold test
                test_tree = ID3DecisionTree(depth_limit=15)
                test_tree.fit(X_train_cv, y_train_cv)
                
                # Save the score for this round
                score = np.mean(test_tree.predict(X_val) == y_val)
                cv_scores.append(score)
            
            # Display the final average of the 5 tests
            avg_cv = np.mean(cv_scores) * 100
            print(f"Final Cross-Validation Mean: {avg_cv:.2f}%")
            
        except Exception as e:
            print(f"Error reading CSV or training model: {e}")

    def get_move(self, state):
        """
        Translates the bitboard state and asks the tree for a prediction.

        Args:
            state (Position): The current state of the game board.

        Returns:
            int: The index of the predicted move.
        """
        # The game uses bits, but the tree needs simple numbers (-1, 0, 1)
        simple_board = np.full(42, -1)
        
        p0 = state.position if state.turn == 0 else (state.position ^ state.mask)
        p1 = state.position if state.turn == 1 else (state.position ^ state.mask)
        
        for i in range(42):
            bit_mask = 1 << ((i % 7) * 7 + (i // 7))
            if p0 & bit_mask: simple_board[i] = 0
            elif p1 & bit_mask: simple_board[i] = 1
        
        input_data = np.append(simple_board, state.turn)
        
        predicted_move = int(self.tree.predict(np.array([input_data]))[0])
        
        legal_moves = state.legal_moves()
        if predicted_move in legal_moves:
            return predicted_move
        else:
            return random.choice(legal_moves)